[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/C.%20Transportation%2C%20Routing%20%26%20Freight%20Management%20%28The%20Physical%20Internet%29/079.%20The%20Freight%20Carrier%20Selection%20%26%20Bidding%20Problem/P79-Tier-2_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---



# 79. The Freight Carrier Selection and Bidding Problem
## Tier 2: Greedy Heuristic Implementation

### Purpose and scope
This notebook assigns freight lanes to carriers using a greedy heuristic. It processes higher-demand lanes first and, for each lane, assigns the available carrier with the highest weighted composite score. A carrier is available for a lane only when its remaining capacity can accommodate that lane's full demand.

The inputs are carrier and lane names, carrier-lane bid prices and service scores, carrier reliability scores and capacity limits, lane demands, and three score weights. The outputs are lane assignments, carrier utilization, total assigned cost, average reliability across assigned lanes, total service score, and measured execution time. A cost-only reference assignment, visualizations, and synthetic scalability tests provide additional context.

### Model assumptions
- Each lane is assigned to at most one carrier; a lane is either assigned in full or left unassigned.
- A carrier's capacity is shared across its assigned lanes, and lane demand consumes that capacity.
- Bid price is a cost per unit of lane demand. Reliability is specific to the carrier, while service quality is specific to a carrier-lane pair.
- The weighted score is a local selection rule, not a global objective optimized over all assignments. The reported total cost, average reliability, and total service score are separate descriptive metrics.
- The example does not model split shipments, time windows, uncertain demand, contractual commitments, or interactions between different lanes beyond shared carrier capacity.

### Required background: scores and greedy choices
A score combines quantities on different scales into one ranking value. In this implementation, cost is represented by a scaled reciprocal, so lower bids receive higher cost scores; reliability and service scores are divided by 100. The weights determine the relative contribution of these three terms. This is a specified weighted score, not a min-max normalization of the criteria.

A greedy algorithm makes the best available local choice at each step according to its ranking rule. Here, the lane order is fixed by decreasing demand, and each lane is assigned to the currently highest-scoring carrier that has enough remaining capacity. This step-by-step rule is fast and transparent, but a locally best choice can use capacity that would have been more valuable for another lane. Therefore, the heuristic does not guarantee a globally optimal assignment.

### Mathematical model used by the heuristic
Let $C$ be the set of carriers and $L$ the set of lanes. For carrier $c \in C$ and lane $l \in L$, let $b_{cl}$ be the bid price, $r_c$ the reliability score, and $s_{cl}$ the service score. Let $d_l$ be lane demand, $K_c$ carrier capacity, and $u_c$ the capacity used so far. The weights are $\alpha$, $\beta$, and $\gamma$.

The score used to rank feasible carriers for lane $l$ is

$$\operatorname{score}_{cl} = \alpha \frac{1000}{b_{cl}} + \beta \frac{r_c}{100} + \gamma \frac{s_{cl}}{100}.$$

A carrier is feasible for the current lane when

$$u_c + d_l \leq K_c.$$

Lanes are considered in non-increasing order of $d_l$. For each lane, the heuristic chooses a feasible carrier with the largest score, assigns the lane, and updates that carrier's used capacity by $d_l$. If no carrier is feasible, the lane remains unassigned. Ties in score are resolved by carrier order because the implementation uses a stable sort and retains the first highest-scoring candidate.

In the code, `carrier_scores` stores feasible carrier-score pairs, `lane_order` determines the demand-based processing sequence, `assignment` records the chosen carrier index (or `-1`), and `carrier_utilization` stores each $u_c$. For $|L|$ lanes and $|C|$ carriers, scoring requires $O(|L||C|)$ work; sorting the feasible carriers lane by lane gives a worst-case time bound of $O(|L||C|\log |C|)$. The stored assignment and utilization require $O(|L|+|C|)$ space.

### How to interpret the results
The assignment and capacity chart show which lanes were placed and how much capacity each carrier uses. The total cost is the sum of bid price times demand for assigned lanes only. Average reliability is the unweighted mean of the assigned carriers' reliability scores, one score per assigned lane; total service is the sum of service scores for assigned carrier-lane pairs. These metrics have different units and should not be treated as interchangeable measures of solution quality.

The cost-only reference later in the notebook is a feasible, minimum-cost assignment for this particular data set. It does not optimize the weighted score or establish that the greedy heuristic is optimal for a multi-criteria problem. Runtime measurements are specific to the machine and execution environment and should not be interpreted as a general speed guarantee.

In [ ]:
# Import the libraries used for data handling, timing, and visualization.
import time as time_module

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Keep plots visually consistent across the notebook.
plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

# Fix the random seed so the synthetic scalability cases are reproducible.
np.random.seed(42)

In [ ]:
# Define the freight carrier selection problem data.

# Carriers and lanes
carriers = ["Carrier 1", "Carrier 2", "Carrier 3"]
lanes = ["Lane 1 (NYC-CHI)", "Lane 2 (LAX-DAL)"]

# Bid prices are costs per unit of lane demand.
bid_prices = np.array([
    [500, 450],  # Carrier 1 bids
    [480, 470],  # Carrier 2 bids
    [520, 440],  # Carrier 3 bids
])

# Reliability scores are on a 0-100 scale and are carrier-specific.
reliability_scores = np.array([95, 88, 92])

# Service scores are on a 0-100 scale for each carrier-lane pair.
service_scores = np.array([
    [90, 85],  # Carrier 1 service scores
    [88, 92],  # Carrier 2 service scores
    [87, 90],  # Carrier 3 service scores
])

# Demand is measured in capacity units for each lane.
demand = np.array([100, 80])

# Capacity limits are measured in the same units as demand.
capacity_limits = np.array([150, 120, 200])

# Weights for cost, reliability, and service components of the score.
alpha = 0.6
beta = 0.3
gamma = 0.1

print("Problem data:")
print(f"Carriers: {carriers}")
print(f"Lanes: {lanes}")
print(f"Demand: {demand}")
print(f"Capacity limits: {capacity_limits}")
print(f"Score weights (alpha, beta, gamma): ({alpha}, {beta}, {gamma})")

In [ ]:
# Apply the demand-first greedy assignment rule and report its metrics.

def greedy_carrier_selection(
    carriers,
    lanes,
    bid_prices,
    reliability_scores,
    service_scores,
    demand,
    capacity_limits,
    alpha,
    beta,
    gamma,
):
    """Assign lanes in descending demand order using the weighted score."""
    num_carriers = len(carriers)
    num_lanes = len(lanes)

    if bid_prices.shape != (num_carriers, num_lanes):
        raise ValueError("bid_prices must have one row per carrier and one column per lane.")
    if service_scores.shape != (num_carriers, num_lanes):
        raise ValueError("service_scores must have one row per carrier and one column per lane.")
    if reliability_scores.shape != (num_carriers,):
        raise ValueError("reliability_scores must contain one score per carrier.")
    if demand.shape != (num_lanes,):
        raise ValueError("demand must contain one value per lane.")
    if capacity_limits.shape != (num_carriers,):
        raise ValueError("capacity_limits must contain one limit per carrier.")
    if np.any(bid_prices <= 0):
        raise ValueError("Bid prices must be positive because the score uses their reciprocals.")

    start_time = time_module.perf_counter()

    # The assignment list uses -1 to distinguish an unassigned lane from carrier index 0.
    assignment = [-1] * num_lanes
    carrier_utilization = np.zeros(num_carriers, dtype=float)

    # Serving larger demands first gives those lanes priority for scarce capacity.
    lane_order = np.argsort(-demand)

    for lane_idx in lane_order:
        carrier_scores = []

        for carrier_idx in range(num_carriers):
            # A candidate is considered only if its remaining capacity covers the full lane.
            if carrier_utilization[carrier_idx] + demand[lane_idx] <= capacity_limits[carrier_idx]:
                cost = bid_prices[carrier_idx, lane_idx]
                reliability = reliability_scores[carrier_idx]
                service = service_scores[carrier_idx, lane_idx]

                # The reciprocal makes a lower bid contribute a higher cost score.
                score = (
                    alpha * (1 / cost) * 1000
                    + beta * reliability / 100
                    + gamma * service / 100
                )
                carrier_scores.append((carrier_idx, score))

        # Stable descending sorting makes carrier order the tie-break rule.
        carrier_scores.sort(key=lambda item: item[1], reverse=True)

        if carrier_scores:
            best_carrier, _ = carrier_scores[0]
            assignment[lane_idx] = best_carrier
            carrier_utilization[best_carrier] += demand[lane_idx]

    # Compute descriptive metrics from assigned lanes only.
    total_cost = 0.0
    total_reliability = 0.0
    total_service = 0.0
    assigned_lanes = 0

    for lane_idx, carrier_idx in enumerate(assignment):
        if carrier_idx != -1:
            total_cost += bid_prices[carrier_idx, lane_idx] * demand[lane_idx]
            total_reliability += reliability_scores[carrier_idx]
            total_service += service_scores[carrier_idx, lane_idx]
            assigned_lanes += 1

    avg_reliability = (
        total_reliability / assigned_lanes if assigned_lanes > 0 else 0.0
    )
    execution_time = time_module.perf_counter() - start_time

    # These checks verify feasibility and consistency of the reported aggregate metrics.
    assert np.all(carrier_utilization <= capacity_limits)
    assert assigned_lanes == sum(carrier_idx != -1 for carrier_idx in assignment)

    return {
        "assignment": assignment,
        "carrier_utilization": carrier_utilization,
        "total_cost": total_cost,
        "avg_reliability": avg_reliability,
        "total_service": total_service,
        "assigned_lanes": assigned_lanes,
        "execution_time": execution_time,
    }


greedy_solution = greedy_carrier_selection(
    carriers=carriers,
    lanes=lanes,
    bid_prices=bid_prices,
    reliability_scores=reliability_scores,
    service_scores=service_scores,
    demand=demand,
    capacity_limits=capacity_limits,
    alpha=alpha,
    beta=beta,
    gamma=gamma,
)

print("Greedy heuristic results:")
for lane_idx in np.argsort(-demand):
    carrier_idx = greedy_solution["assignment"][lane_idx]
    if carrier_idx != -1:
        print(f"{lanes[lane_idx]}: {carriers[carrier_idx]}")
    else:
        print(f"{lanes[lane_idx]}: Unassigned")

print(f"\nTotal assigned cost: ${greedy_solution['total_cost']:,.2f}")
print(f"Average reliability: {greedy_solution['avg_reliability']:.1f}%")
print(f"Total service score: {greedy_solution['total_service']:.1f}")
print(f"Assigned lanes: {greedy_solution['assigned_lanes']}/{len(lanes)}")
print(f"Measured execution time: {greedy_solution['execution_time']:.6f} seconds")

In [ ]:
# Visualize runtime, capacity use, reported metrics, and lane assignments.

fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle("Greedy Heuristic: Freight Carrier Selection", fontsize=16, fontweight="bold")

# Runtime is shown as a single measured value, not as a comparison with an estimate.
ax1 = axes[0, 0]
ax1.bar(
    ["Greedy heuristic"],
    [greedy_solution["execution_time"]],
    color="lightgreen",
    alpha=0.8,
)
ax1.set_ylabel("Execution time (seconds)")
ax1.set_title("Measured Runtime")
ax1.grid(True, axis="y", alpha=0.3)
ax1.text(
    0,
    greedy_solution["execution_time"],
    f"{greedy_solution['execution_time']:.6f} s",
    ha="center",
    va="bottom",
    fontweight="bold",
)

# Used capacity and total capacity share the same unit: demand units.
ax2 = axes[0, 1]
x_pos = np.arange(len(carriers))
width = 0.35
ax2.bar(
    x_pos - width / 2,
    greedy_solution["carrier_utilization"],
    width,
    label="Used capacity",
    color="lightgreen",
    alpha=0.8,
)
ax2.bar(
    x_pos + width / 2,
    capacity_limits,
    width,
    label="Capacity limit",
    color="lightcoral",
    alpha=0.8,
)
ax2.set_xlabel("Carrier")
ax2.set_ylabel("Capacity (demand units)")
ax2.set_title("Carrier Capacity Utilization")
ax2.set_xticks(x_pos)
ax2.set_xticklabels(carriers)
ax2.legend()
ax2.grid(True, axis="y", alpha=0.3)

for i, (used, capacity) in enumerate(
    zip(greedy_solution["carrier_utilization"], capacity_limits)
):
    if used > 0:
        percentage = used / capacity * 100
        ax2.text(i - width / 2, used, f"{percentage:.1f}%", ha="center", va="bottom")

# These metrics have different units; the bars are descriptive, not directly comparable.
ax3 = axes[1, 0]
metric_names = ["Total cost ($)", "Average reliability (%)", "Total service score"]
metric_values = [
    greedy_solution["total_cost"],
    greedy_solution["avg_reliability"],
    greedy_solution["total_service"],
]
metric_bars = ax3.bar(
    metric_names,
    metric_values,
    color=["lightcoral", "lightgreen", "lightblue"],
    alpha=0.8,
)
ax3.set_ylabel("Reported value (metric-specific units)")
ax3.set_title("Solution Metrics: Different Units")
ax3.tick_params(axis="x", rotation=25)
ax3.grid(True, axis="y", alpha=0.3)
for metric_idx, bar in enumerate(metric_bars):
    value = metric_values[metric_idx]
    label = f"${value:,.0f}" if metric_idx == 0 else f"{value:.1f}"
    ax3.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        label,
        ha="center",
        va="bottom",
        fontweight="bold",
    )

# The binary heatmap marks whether each carrier-lane pair is assigned.
ax4 = axes[1, 1]
assignment_matrix = np.zeros((len(carriers), len(lanes)), dtype=int)
for lane_idx, carrier_idx in enumerate(greedy_solution["assignment"]):
    if carrier_idx != -1:
        assignment_matrix[carrier_idx, lane_idx] = 1

lane_labels = [lane.split("(")[1].split(")")[0] for lane in lanes]
sns.heatmap(
    assignment_matrix,
    annot=True,
    fmt="d",
    cmap="Greens",
    vmin=0,
    vmax=1,
    xticklabels=lane_labels,
    yticklabels=carriers,
    cbar=True,
    cbar_kws={"ticks": [0, 1], "label": "Assignment (0 = no, 1 = yes)"},
    ax=ax4,
)
ax4.set_title("Lane Assignment Matrix")
ax4.set_xlabel("Lane")
ax4.set_ylabel("Carrier")

plt.tight_layout()
plt.show()

In [ ]:
# Compare the heuristic with a feasible cost-only reference assignment.

def solve_optimal_for_comparison(bid_prices, demand, reliability_scores, service_scores, capacity_limits):
    """Return the cost-minimizing assignment for the supplied two-lane example."""
    # For this example, each lane's lowest-bid carrier can serve both lanes together.
    best_assignment = [1, 2]

    if len(best_assignment) != len(demand):
        raise ValueError("The reference assignment must contain one carrier per lane.")

    # Confirm that the selected carrier is a minimum-bid option for each lane.
    for lane_idx, carrier_idx in enumerate(best_assignment):
        if not np.isclose(bid_prices[carrier_idx, lane_idx], np.min(bid_prices[:, lane_idx])):
            raise ValueError("The reference assignment must use a minimum-bid carrier for each lane.")

    reference_utilization = np.zeros(len(capacity_limits), dtype=float)
    for lane_idx, carrier_idx in enumerate(best_assignment):
        reference_utilization[carrier_idx] += demand[lane_idx]
    if np.any(reference_utilization > capacity_limits):
        raise ValueError("The cost-only reference assignment violates a capacity limit.")

    total_cost = sum(
        bid_prices[carrier_idx, lane_idx] * demand[lane_idx]
        for lane_idx, carrier_idx in enumerate(best_assignment)
    )
    avg_reliability = np.mean(
        [reliability_scores[carrier_idx] for carrier_idx in best_assignment]
    )
    total_service = sum(
        service_scores[carrier_idx, lane_idx]
        for lane_idx, carrier_idx in enumerate(best_assignment)
    )

    return {
        "assignment": best_assignment,
        "total_cost": total_cost,
        "avg_reliability": avg_reliability,
        "total_service": total_service,
    }


reference_solution = solve_optimal_for_comparison(
    bid_prices=bid_prices,
    demand=demand,
    reliability_scores=reliability_scores,
    service_scores=service_scores,
    capacity_limits=capacity_limits,
)

cost_difference = greedy_solution["total_cost"] - reference_solution["total_cost"]
reliability_difference = (
    greedy_solution["avg_reliability"] - reference_solution["avg_reliability"]
)
service_difference = greedy_solution["total_service"] - reference_solution["total_service"]

cost_ratio = (
    greedy_solution["total_cost"] / reference_solution["total_cost"]
    if reference_solution["total_cost"] != 0
    else np.nan
)
reliability_ratio = (
    greedy_solution["avg_reliability"] / reference_solution["avg_reliability"]
    if reference_solution["avg_reliability"] != 0
    else np.nan
)

print("Greedy heuristic versus the cost-only reference")
print("=" * 52)
print("\nCost comparison:")
print(f"  Cost-only reference: ${reference_solution['total_cost']:,.2f}")
print(f"  Greedy heuristic:    ${greedy_solution['total_cost']:,.2f}")
print(f"  Difference:          ${cost_difference:+,.2f}")
if np.isfinite(cost_ratio):
    print(f"  Greedy/reference cost ratio: {cost_ratio:.2%}")

print("\nReliability comparison (unweighted mean across assigned lanes):")
print(f"  Cost-only reference: {reference_solution['avg_reliability']:.1f}%")
print(f"  Greedy heuristic:    {greedy_solution['avg_reliability']:.1f}%")
print(f"  Difference:          {reliability_difference:+.1f} percentage points")
if np.isfinite(reliability_ratio):
    print(f"  Greedy/reference reliability ratio: {reliability_ratio:.2%}")

print("\nService comparison (sum of assigned lane scores):")
print(f"  Cost-only reference: {reference_solution['total_service']:.1f}")
print(f"  Greedy heuristic:    {greedy_solution['total_service']:.1f}")
print(f"  Difference:          {service_difference:+.1f}")

print("\nThe reference minimizes bid cost for this data set; it does not optimize reliability or service.")

In [ ]:
# Test how the same demand-first greedy rule behaves on reproducible synthetic sizes.

def test_scalability():
    """Measure the heuristic on randomly generated carrier-lane instances."""
    test_sizes = [
        (3, 2),
        (5, 3),
        (10, 5),
        (20, 10),
    ]

    results = []

    for num_carriers, num_lanes in test_sizes:
        # Resetting the seed gives each test size a reproducible data generation sequence.
        np.random.seed(42)

        test_bid_prices = np.random.uniform(400, 600, (num_carriers, num_lanes))
        test_reliability = np.random.uniform(80, 95, num_carriers)
        test_service = np.random.uniform(80, 95, (num_carriers, num_lanes))
        test_demand = np.random.uniform(50, 150, num_lanes).astype(int)
        test_capacity = np.random.uniform(100, 300, num_carriers).astype(int)

        start_time = time_module.perf_counter()
        assignment = [-1] * num_lanes
        utilization = np.zeros(num_carriers, dtype=float)

        # Use the same lane priority rule as the main example.
        lane_order = np.argsort(-test_demand)
        for lane_idx in lane_order:
            carrier_scores = []

            for carrier_idx in range(num_carriers):
                if utilization[carrier_idx] + test_demand[lane_idx] <= test_capacity[carrier_idx]:
                    score = (
                        0.6 * (1 / test_bid_prices[carrier_idx, lane_idx]) * 1000
                        + 0.3 * test_reliability[carrier_idx] / 100
                        + 0.1 * test_service[carrier_idx, lane_idx] / 100
                    )
                    carrier_scores.append((carrier_idx, score))

            carrier_scores.sort(key=lambda item: item[1], reverse=True)
            if carrier_scores:
                best_carrier = carrier_scores[0][0]
                assignment[lane_idx] = best_carrier
                utilization[best_carrier] += test_demand[lane_idx]

        execution_time = time_module.perf_counter() - start_time
        assigned_lanes = sum(carrier_idx != -1 for carrier_idx in assignment)

        # Feasibility checks ensure the scalability measurements reflect valid assignments.
        assert np.all(utilization <= test_capacity)
        assert assigned_lanes == sum(carrier_idx != -1 for carrier_idx in assignment)

        results.append({
            "carriers": num_carriers,
            "lanes": num_lanes,
            "execution_time": execution_time,
            "assigned_lanes": assigned_lanes,
            "total_lanes": num_lanes,
        })

    return pd.DataFrame(results)


scalability_results = test_scalability()

print("Scalability analysis results:")
print(scalability_results.to_string(index=False))

# Plot measured runtime and the share of lanes assigned in each synthetic case.
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle("Greedy Heuristic: Synthetic Scalability Tests", fontsize=16, fontweight="bold")

problem_sizes = scalability_results["carriers"] * scalability_results["lanes"]
ax1.plot(
    problem_sizes,
    scalability_results["execution_time"],
    "o-",
    linewidth=2,
    markersize=8,
)
ax1.set_xlabel("Carrier-lane pairs (carriers × lanes)")
ax1.set_ylabel("Measured execution time (seconds)")
ax1.set_title("Runtime by Test Size")
ax1.grid(True, alpha=0.3)
ax1.set_yscale("log")

success_rates = (
    scalability_results["assigned_lanes"]
    / scalability_results["total_lanes"]
    * 100
)
ax2.bar(
    range(len(scalability_results)),
    success_rates,
    color="lightgreen",
    alpha=0.8,
)
ax2.set_xlabel("Test case (carriers × lanes)")
ax2.set_ylabel("Assigned lanes (%)")
ax2.set_title("Assignment Completion Rate")
ax2.set_xticks(range(len(scalability_results)))
ax2.set_xticklabels(
    [f"{row['carriers']}×{row['lanes']}" for _, row in scalability_results.iterrows()],
        rotation=45,
)
ax2.grid(True, axis="y", alpha=0.3)
ax2.set_ylim(0, 100)

for i, success_rate in enumerate(success_rates):
    ax2.text(i, success_rate + 2, f"{success_rate:.0f}%", ha="center", va="bottom")

plt.tight_layout()
plt.show()

## Why use a greedy tier?
The greedy heuristic is useful when a clear, quick assignment rule is valuable. Its work is based on scoring carriers for each lane rather than searching through every possible complete assignment. The demand-first ordering and capacity checks make the decisions operationally interpretable, while the score gives a consistent way to rank feasible carriers.

The method's speed and simplicity come with a trade-off: it commits to each assignment without evaluating every downstream consequence. A locally highest score may consume capacity needed by a later lane, and the weight choices affect the ranking. There is no general guarantee that the resulting assignment globally minimizes cost or maximizes a combined measure of cost, reliability, and service.

### Strengths and limitations
**Strengths**
- Simple decision logic that is easy to inspect and implement.
- Capacity feasibility is checked before every assignment.
- The same rule can be applied to larger carrier-lane tables.
- The resulting choices and their metrics can be reviewed directly.

**Limitations**
- The order of lanes and the score weights can affect the result.
- A locally best choice can prevent a better overall assignment.
- Unassigned lanes are possible when remaining capacities are insufficient.
- Runtime measurements depend on hardware, software, and problem data.

### When this tier can be useful
This approach can be useful for rapid dispatch decisions, larger instances where a quick feasible assignment is required, or settings where decision makers value a transparent rule. Its output should be interpreted as the result of the stated heuristic, not as a guarantee of global optimality. The synthetic scalability experiment illustrates runtime and assignment completion for its generated cases; it does not establish performance for every real freight network.

In [ ]:
# Summarize the assignment, its measured metrics, and the heuristic's interpretation.
print("=" * 70)
print("GREEDY HEURISTIC: FINAL SUMMARY")
print("=" * 70)

print("\nASSIGNMENTS:")
for lane_idx in np.argsort(-demand):
    carrier_idx = greedy_solution["assignment"][lane_idx]
    if carrier_idx != -1:
        print(f"  {lanes[lane_idx]}: {carriers[carrier_idx]}")
    else:
        print(f"  {lanes[lane_idx]}: Unassigned")

print("\nMETRICS:")
print(f"  Total assigned cost: ${greedy_solution['total_cost']:,.2f}")
print(f"  Average reliability: {greedy_solution['avg_reliability']:.1f}%")
print(f"  Total service score: {greedy_solution['total_service']:.1f}")
print(f"  Assigned lanes: {greedy_solution['assigned_lanes']}/{len(lanes)}")
print(f"  Measured execution time: {greedy_solution['execution_time']:.6f} seconds")

print("\nMETHOD INTERPRETATION:")
print("  Lanes are considered in descending order of demand.")
print("  Only carriers with enough remaining capacity are considered.")
print("  The highest weighted score among feasible carriers is selected.")
print("  The heuristic does not guarantee a globally optimal assignment.")

print("\nCOST-ONLY REFERENCE COMPARISON:")
print(f"  Cost difference (greedy minus reference): ${cost_difference:+,.2f}")
print(f"  Reliability difference: {reliability_difference:+.1f} percentage points")
print(f"  Service-score difference: {service_difference:+.1f}")
print("  The reference is cost-minimizing for this example and is not a multi-criteria optimum.")

print("\nUse the assignment and capacity checks to assess feasibility, and interpret each quality metric")
print("according to its own definition and units.")
print("=" * 70)